## 1. CARGA DE DATASET (DATA)

In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/DETALLE_INVERSIONES.csv")
pd.options.display.float_format = '{:.2f}'.format
data.shape

(259728, 68)

## 2. CONTEXTUALIZACIÓN 2018-2025

In [2]:
data["FECHA_VIABILIDAD"].unique() 

<StringArray>
['2026-02-19 00:00:00', '2026-01-08 00:00:00', '2014-05-26 00:00:00',
 '2020-12-03 00:00:00', '2018-04-29 00:00:00', '2011-07-14 00:00:00',
 '2008-02-13 00:00:00', '2015-10-07 00:00:00', '2012-05-11 00:00:00',
 '2011-07-25 00:00:00',
 ...
 '2006-01-13 00:00:00', '2004-12-02 00:00:00', '2008-03-16 00:00:00',
 '2004-11-03 00:00:00', '2004-07-08 00:00:00', '2003-08-04 00:00:00',
 '2010-07-10 00:00:00', '2012-10-14 00:00:00', '2002-11-20 00:00:00',
 '2003-02-17 00:00:00']
Length: 7662, dtype: str

In [3]:
data["FECHA_VIABILIDAD"] = pd.to_datetime(data["FECHA_VIABILIDAD"], errors='coerce')
data = data[data["FECHA_VIABILIDAD"].dt.year.between(2018, 2025)]
print(f"Registros en el periodo 2018-2025: {len(data)}")

Registros en el periodo 2018-2025: 156161


**El periodo segmentado es respecto a la viabilidad de los proyectos.**

## 3. LIMPIEZA

3.1 VARIABLES RELEVANTES

In [4]:
cols = [

# Ejecución
"AVANCE_FISICO", # Porcentaje de avance físico de inversión
"AVANCE_EJECUCION", # Porcentaje de avance de ejecución general de la inversión

# Financieras
"MONTO_VIABLE", # Monto viable de la inversión
"COSTO_ACTUALIZADO", # Costo actualizado de la inversión
"MONTO_LAUDO", # Costo de controversias
"PIM_ANIO_ACTUAL", # Monto asignado en el PIM del año actual
"SALDO_EJECUTAR", # Monto de saldo pendiente de ejecutar para el siguiente año
"MONTO_ET_F8", # Monto registrado en la etapa de elaboración de ET del F8

# Geográficas
"DEPARTAMENTO", # Departamento de localización de la inversión
"PROVINCIA", # Provincia de localización de la inversión
"DISTRITO", # Distrito de localización de la inversión

# Gestión
"ESTADO", # Estado de la inversión
"SITUACION", # Situación de la inversión
"EXPEDIENTE_TECNICO", # Indica si tiene registro de ET
"TIENE_F9", # Indica si tiene registro de F9
"ETAPA_F9", # Descripción de la etapa vigente en el formato F9
"TIENE_F8", # Indica si tiene registro de F8
"ETAPA_F8", # Descripción de la etapa vigente en el formato F8
"TIENE_F12B", # Indica si tiene registro de F12B
"IND_IOARR_EMERG", # Si tiene IOARR de emergencia

# Identificación
"CODIGO_UNICO", # CUI de la inversión
"NOMBRE_INVERSION", # Nombre de la inversión
"TIPO_INVERSION", # Descripción del tipo de inversión
"DES_MODALIDAD", # Descripción de la modalidad de ejecución
"DES_TIPOLOGIA", # Descripción de la tipología de inversión
"NUM_HABITANTES_BENEF", # Beneficiarios de la ejecución de la inversión

# Institucional
"NIVEL", # Nivel de la fase de FyE
"SECTOR", # Sector de la fase de FyE
"ENTIDAD", # Pliego o entidad de la fase de FyE

# Tiempo
"FECHA_VIABILIDAD", # Fecha de viabilidad o aprobación
"PRIMER_DEVENGADO", # Periodo del primer devengado realizado en el SIAF
"ULTIMO_DEVENGADO", # Periodo del último devengado realizado en el SIAF
"ULT_FEC_DECLA_ESTIM", # Fecha de última declaración de avance físico de la inversión
"FEC_INI_EJECUCION", # Fecha de inicio de ejecución registrada en F8
"FEC_FIN_EJECUCION", # Fecha de fin de ejecución registrada en F8
"FEC_INI_EJEC_FISICA", # Fecha de inicio de ejecución física registrada en F12B
"FEC_FIN_EJEC_FISICA" # Fecha de fin de ejecución física registrada en F12B

]

data = data[cols]

3.2 AJUSTES DE FORMATO

In [5]:
data["FECHA_VIABILIDAD"] = pd.to_datetime(data["FECHA_VIABILIDAD"],errors="coerce")
data["ULT_FEC_DECLA_ESTIM"] = pd.to_datetime(data["ULT_FEC_DECLA_ESTIM"],errors="coerce")
data["FEC_INI_EJECUCION"] = pd.to_datetime(data["FEC_INI_EJECUCION"],errors="coerce")
data["FEC_FIN_EJECUCION"] = pd.to_datetime(data["FEC_FIN_EJECUCION"],errors="coerce")
data["FEC_INI_EJEC_FISICA"] = pd.to_datetime(data["FEC_INI_EJEC_FISICA"],errors="coerce")
data["FEC_FIN_EJEC_FISICA"] = pd.to_datetime(data["FEC_FIN_EJEC_FISICA"],errors="coerce")
data['PRIMER_DEVENGADO'] = pd.to_datetime(data['PRIMER_DEVENGADO'].astype(str),format='%Y%m',errors='coerce')
data['ULTIMO_DEVENGADO'] = pd.to_datetime(data['ULTIMO_DEVENGADO'].astype(str),format='%Y%m',errors='coerce')
data['CODIGO_UNICO'] = data['CODIGO_UNICO'].astype(str)
data['NUM_HABITANTES_BENEF'] = data['NUM_HABITANTES_BENEF'].astype('Int64')
data['PIM_ANIO_ACTUAL'] = data['PIM_ANIO_ACTUAL'].astype(float)

3.3 VERIFICACIÓN DE NULOS Y FALTANTES

In [6]:
perdidos = data.isnull().sum().sort_values(ascending=False)
print(data.shape)
perdidos

(156161, 37)


ULTIMO_DEVENGADO        156161
PRIMER_DEVENGADO        156161
ETAPA_F9                148148
AVANCE_FISICO           106846
ULT_FEC_DECLA_ESTIM      87036
NUM_HABITANTES_BENEF     55554
MONTO_ET_F8              47729
FEC_INI_EJEC_FISICA      44913
FEC_FIN_EJEC_FISICA      44912
AVANCE_EJECUCION         43003
ETAPA_F8                 33400
FEC_INI_EJECUCION        13956
FEC_FIN_EJECUCION        13955
DES_TIPOLOGIA             7547
DISTRITO                    28
PROVINCIA                   18
DEPARTAMENTO                17
DES_MODALIDAD               14
CODIGO_UNICO                 6
TIENE_F8                     0
ENTIDAD                      0
MONTO_VIABLE                 0
ESTADO                       0
SITUACION                    0
COSTO_ACTUALIZADO            0
MONTO_LAUDO                  0
PIM_ANIO_ACTUAL              0
FECHA_VIABILIDAD             0
SECTOR                       0
IND_IOARR_EMERG              0
NIVEL                        0
SALDO_EJECUTAR               0
EXPEDIEN

3.4 CREACIÓN DE VARIABLES

In [12]:
data["ACTAULIZACION"]=(data["COSTO_ACTUALIZADO"]-data["MONTO_VIABLE"])
data["DURACION_PROYECTO"]=(data["FEC_FIN_EJECUCION"]-data["FEC_INI_EJECUCION"]).dt.days
data["DURACION_FISICA"]=(data["FEC_FIN_EJEC_FISICA"]-data["FEC_INI_EJEC_FISICA"]).dt.days
data["MARGEN_FYE_EJEC"]=(data["FEC_INI_EJECUCION"]-data["FECHA_VIABILIDAD"]).dt.days

3.5 VERIFICACIÓN

In [13]:
data.describe()

,AVANCE_FISICO,AVANCE_EJECUCION,MONTO_VIABLE,COSTO_ACTUALIZADO,MONTO_LAUDO,PIM_ANIO_ACTUAL,SALDO_EJECUTAR,MONTO_ET_F8,NUM_HABITANTES_BENEF,FECHA_VIABILIDAD,...,ULTIMO_DEVENGADO,ULT_FEC_DECLA_ESTIM,FEC_INI_EJECUCION,FEC_FIN_EJECUCION,FEC_INI_EJEC_FISICA,FEC_FIN_EJEC_FISICA,ACTAULIZACION,DURACION_PROYECTO,DURACION_FISICA,MARGEN_FYE_EJEC
count,49315.00,113158.00,156161.00,156161.00,156161.00,156161.00,156161.00,108432.00,100607.00,156161,...,0,69125,142205,142206,111248,111249,156161.00,142205.00,111245.00,142205.00
mean,6239.88,8470.11,5517455.57,6364805.43,650.33,234395.52,5227423.43,5818176.23,82667.94,2022-09-29 11:19:56.296130,...,NaT,2025-01-21 10:20:25.082372,2023-10-27 01:36:53.871523,2023-12-13 17:42:55.941943,2023-10-09 05:32:07.887243,2025-01-17 20:07:22.294312,847349.86,47.67,466.58,410.18
min,0.00,0.00,1.00,0.01,0.00,0.00,-19393646.45,0.00,0.00,2018-01-02 00:00:00,...,NaT,2020-06-01 10:54:12,2002-07-13 00:00:00,2002-07-13 00:00:00,1969-12-31 00:00:00,2002-11-29 00:00:00,-846032640.50,-3462.00,-7243.00,-7170.00
25%,45.02,0.00,296065.18,326770.37,0.00,0.00,22690.06,66509.38,172.00,2020-12-02 00:00:00,...,NaT,2024-01-16 09:24:55,2022-05-04 00:00:00,2022-06-12 00:00:00,2022-03-01 00:00:00,2022-12-01 00:00:00,0.00,0.00,63.00,64.00
50%,93.88,1.17,877489.72,984217.85,0.00,0.00,436000.00,543637.45,509.00,2023-03-24 00:00:00,...,NaT,2026-01-21 14:32:07,2024-03-01 00:00:00,2024-05-01 00:00:00,2024-02-05 00:00:00,2025-01-30 00:00:00,0.00,0.00,149.00,181.00
75%,100.00,85.96,2817327.05,3194777.45,0.00,0.00,2141379.67,2574562.77,2024.00,2024-07-31 00:00:00,...,NaT,2026-02-24 13:49:57,2025-08-01 00:00:00,2025-09-29 00:00:00,2025-08-13 00:00:00,2026-06-29 00:00:00,112898.33,0.00,451.00,545.00
max,217378466.49,717320069.10,30552699580.07,30527140923.70,15845848.21,500000000.00,30552699580.07,29171496059.60,856926112.00,2025-12-31 00:00:00,...,NaT,2026-03-24 22:39:50,2040-09-01 00:00:00,2041-08-01 00:00:00,2040-09-01 00:00:00,2066-05-02 00:00:00,4692615167.44,6940.00,19966.00,6295.00
std,1020488.85,2217461.49,152540228.00,154086225.71,59680.14,2559399.68,152982895.74,165804567.42,5499375.45,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,17807367.41,162.42,838.36,569.20


3.6 CORRECCIÓN DE VALORES ATÍPICOS

In [15]:
data = data[data["AVANCE_FISICO"] <= 100]
data = data[data["AVANCE_EJECUCION"] <= 100]
data.describe()

,AVANCE_FISICO,AVANCE_EJECUCION,MONTO_VIABLE,COSTO_ACTUALIZADO,MONTO_LAUDO,PIM_ANIO_ACTUAL,SALDO_EJECUTAR,MONTO_ET_F8,NUM_HABITANTES_BENEF,FECHA_VIABILIDAD,...,ULTIMO_DEVENGADO,ULT_FEC_DECLA_ESTIM,FEC_INI_EJECUCION,FEC_FIN_EJECUCION,FEC_INI_EJEC_FISICA,FEC_FIN_EJEC_FISICA,ACTAULIZACION,DURACION_PROYECTO,DURACION_FISICA,MARGEN_FYE_EJEC
count,49085.00,49085.00,49085.00,49085.00,49085.00,49085.00,49085.00,49084.00,29014.00,49085,...,0,49085,49081,49081,49083,49084,49085.00,49081.00,49083.00,49081.00
mean,71.48,70.94,3082069.59,4453313.16,1623.35,430258.88,1817361.16,3593758.06,90971.70,2022-03-28 09:14:28.004481,...,NaT,2024-11-26 14:15:13.740511,2023-06-08 05:46:52.681078,2023-06-08 05:54:03.968134,2023-06-23 11:41:07.257502,2024-06-23 14:16:39.233966,1371243.58,0.00,366.09,436.90
min,0.00,0.00,1000.00,2492.82,0.00,0.00,-19393646.45,0.62,0.00,2018-01-03 00:00:00,...,NaT,2020-06-01 10:54:12,2002-07-13 00:00:00,2002-07-13 00:00:00,1969-12-31 00:00:00,2003-01-22 00:00:00,-846032640.50,0.00,-7243.00,-7170.00
25%,44.89,45.14,236004.26,270874.39,0.00,0.00,3000.00,48948.95,199.00,2020-07-30 00:00:00,...,NaT,2024-01-04 12:04:30,2022-02-08 00:00:00,2022-02-08 00:00:00,2022-03-02 00:00:00,2022-12-15 00:00:00,-0.03,0.00,62.00,74.00
50%,93.68,91.22,639528.35,768752.99,0.00,0.00,33841.39,402968.29,576.50,2022-05-30 00:00:00,...,NaT,2025-11-04 10:55:56,2023-09-04 00:00:00,2023-09-04 00:00:00,2023-09-19 00:00:00,2024-08-24 00:00:00,45363.06,0.00,143.00,257.00
75%,100.00,99.28,1921580.26,2468256.67,0.00,5267.00,347619.32,1891549.18,2031.00,2023-10-25 00:00:00,...,NaT,2026-02-23 15:33:37,2025-01-25 00:00:00,2025-01-25 00:00:00,2025-02-10 00:00:00,2025-12-29 00:00:00,421446.60,0.00,418.00,628.00
max,100.00,100.00,1479632983.95,4727617513.44,15845848.21,500000000.00,2307520964.00,1914547637.00,856926112.00,2025-12-31 00:00:00,...,NaT,2026-03-24 22:39:50,2037-05-05 00:00:00,2037-05-05 00:00:00,2037-05-05 00:00:00,2046-08-15 00:00:00,4692615167.44,153.00,19966.00,6019.00
std,36.90,35.86,17507583.09,33807824.27,100135.20,3939019.33,19408329.11,24338714.79,6447505.75,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,25726871.43,0.81,591.20,538.90


## 4. GUARDADO

In [16]:
data.to_csv("../data/procesado/Det_inv_limpia.csv",index=False)